**Name:** Saad Abdulaziz Alshehri  
**Student ID:** 2240006495

# Transformers


#### A Transformer is a deep learning architecture designed to process sequences of text using self-attention.

- Transformers are the foundation of many modern NLP models, including BERT, GPT, and T5.

- Before Transformers, we used models such as RNNs and LSTMs for NLP tasks.

<img src="https://drive.google.com/thumbnail?id=1unPqGaKftHxp37-F99Bg6hJzFbQdCGC7&sz=w1000" width="700">

In [ ]:
%pip -q install transformers torch matplotlib

## 1. Load a Transformer

We use **DistilBERT**, a  Transformer fine-tuned for positive/negative sentiment classification.

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# Look inside the Transformer architecture.
print("Transformer layers:", model.config.n_layers)
print("Attention heads per layer:", model.config.n_heads)
print("Hidden size:", model.config.dim)

Transformer layers: 6
Attention heads per layer: 12
Hidden size: 768



<img src="https://drive.google.com/thumbnail?id=1v2dRK1KR_ydZOknjj8Xv8_wjOdXleDYs&sz=w1000" width="500">

## 2. Tokenizing the text

In [2]:
sentence = "The movie was not good."

# Tokenization: converts text into token IDs
inputs = tokenizer(sentence, return_tensors="pt")

# [CLS] = start token, [SEP] = end token
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)

Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']


## 3. Embeddings and self-attention

Each token ID becomes a **vector (embedding)**. Inside each Transformer layer, **self-attention** lets a token use information from other tokens in the same sentence.

In [3]:
import torch

# Enable attention weights
model.set_attn_implementation("eager")

with torch.no_grad():
    embeddings = model.distilbert.embeddings(inputs["input_ids"])
    outputs = model(**inputs, output_attentions=True)

# Attention from the first layer and first head
attention = outputs.attentions[0][0, 0]

print("Embeddings shape:", embeddings.shape)
print("Attention shape:", attention.shape)
print("Tokens:", tokens)
print("Attention weights:\n", attention)

Embeddings shape: torch.Size([1, 8, 768])
Attention shape: torch.Size([8, 8])
Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']
Attention weights:
 tensor([[0.1145, 0.0938, 0.0681, 0.0701, 0.0596, 0.0800, 0.1593, 0.3546],
        [0.4147, 0.0793, 0.0481, 0.1018, 0.1084, 0.0805, 0.0789, 0.0883],
        [0.1634, 0.0327, 0.0975, 0.0403, 0.1819, 0.1402, 0.0866, 0.2574],
        [0.2882, 0.1014, 0.0504, 0.0669, 0.2180, 0.1009, 0.1231, 0.0511],
        [0.3612, 0.0501, 0.0696, 0.1200, 0.0642, 0.1304, 0.0953, 0.1092],
        [0.3065, 0.0469, 0.1175, 0.1915, 0.0630, 0.0555, 0.1120, 0.1071],
        [0.3505, 0.0935, 0.0572, 0.0753, 0.0972, 0.0964, 0.1165, 0.1135],
        [0.3513, 0.1019, 0.0318, 0.0541, 0.1036, 0.0607, 0.0755, 0.2210]])


## 4. Classification: From Transformer to prediction

The Transformer produces contextual representations. The **classification head** turns them into two scores (*logits*). **Softmax** converts these into probabilities that sum to 1.

In [4]:
# Get prediction scores from the Transformer
with torch.no_grad():
    logits = model(**inputs).logits

# Convert scores into probabilities
probs = torch.softmax(logits, dim=1)[0]

# Display the prediction
prediction = probs.argmax().item()

print("Prediction:", model.config.id2label[prediction])
print("Confidence:", round(probs[prediction].item(), 3))


Prediction: NEGATIVE
Confidence: 1.0



### Task: Exploring Transformer Layers and Attention Heads

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

1. Write code to print the number of Transformer layers and the number of attention heads per layer.
2. Calculate and print the total number of attention heads across all Transformer layers.
3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.
4. Explain what the rows and columns of the attention matrix represent.

In [5]:
from transformers import AutoTokenizer, AutoModel

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name, attn_implementation="eager")
bert_model.eval()

# 1. number of layers and attention heads per layer
print("Transformer layers:", bert_model.config.num_hidden_layers)
print("Attention heads per layer:", bert_model.config.num_attention_heads)

# 2. total number of attention heads
total_heads = bert_model.config.num_hidden_layers * bert_model.config.num_attention_heads
print("Total attention heads:", total_heads)

Transformer layers: 12
Attention heads per layer: 12
Total attention heads: 144


In [6]:
import pandas as pd

# 3. attention weights from the third layer and the second head
sentence = "Transformers use self-attention to understand language."

bert_inputs = bert_tokenizer(sentence, return_tensors="pt")
bert_tokens = bert_tokenizer.convert_ids_to_tokens(bert_inputs["input_ids"][0])

with torch.no_grad():
    bert_outputs = bert_model(**bert_inputs, output_attentions=True)

# the index starts from 0, so third layer = 2 and second head = 1
attention = bert_outputs.attentions[2][0, 1]

print("Tokens:", bert_tokens)
print("Attention shape:", attention.shape)
print()
print(pd.DataFrame(attention.numpy().round(3), index=bert_tokens, columns=bert_tokens).to_string())
print()
print("Sum of each row:", attention.sum(dim=1).numpy().round(3))

Tokens: ['[CLS]', 'transformers', 'use', 'self', '-', 'attention', 'to', 'understand', 'language', '.', '[SEP]']
Attention shape: torch.Size([11, 11])

              [CLS]  transformers    use   self      -  attention     to  understand  language      .  [SEP]
[CLS]         0.732         0.008  0.011  0.006  0.017      0.007  0.010       0.012     0.006  0.039  0.155
transformers  0.889         0.005  0.034  0.000  0.003      0.005  0.001       0.001     0.000  0.005  0.057
use           0.143         0.569  0.005  0.035  0.041      0.068  0.090       0.000     0.005  0.009  0.034
self          0.340         0.073  0.002  0.000  0.202      0.081  0.064       0.026     0.002  0.028  0.183
-             0.143         0.070  0.071  0.639  0.012      0.013  0.003       0.003     0.014  0.009  0.022
attention     0.113         0.545  0.012  0.020  0.111      0.010  0.072       0.000     0.000  0.016  0.100
to            0.084         0.022  0.622  0.008  0.009      0.194  0.011       0.007 

**4. What do the rows and columns of the attention matrix represent?**

The attention matrix is 11 x 11 because the sentence has 11 tokens (including `[CLS]` and `[SEP]`).

- **Rows:** each row is a token that is looking at the other tokens (the token that is being processed).
- **Columns:** each column is a token that is being looked at.
- **Each value:** how much attention the row token gives to the column token. Each row sums to 1 because of softmax.

For example, the row "attention" and the column "transformers" is 0.545, so the token "attention" pays a lot of attention to "transformers". Also the row "transformers" gives 0.889 to `[CLS]`, so many tokens in this head mostly look at the `[CLS]` token.
